<a href="https://colab.research.google.com/github/redinbolab/iterative_linear_fit_notebook/blob/main/Iterative_Linear_Fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
###This code was written and edited by Joshua J. Sekela, Ben Creekmore, and Joshua B. Simpson.
###This code is owned exclusively by the laboratory of Matthew Redinbo at UNC-Chapel Hill. https://www.redinbolab.org/.
###This code is intended and may be used only for academic research.
###If issues running this code arise, please direct all questions and concerns to redinbo@unc.edu.
###If data collected using this code is intended for publication, please inquire at redinbo@unc.edu for proper citation.

#@title example input
data = """
Time,0,5,10,15,20
Sample X1,-7.49496E-05,0.071991929,0.10427789,0.173462093,0.235727875
Sample X2,-0.00468723,0.065073508,0.123303546,0.174038628,0.22246757
Sample X3,-0.00929951,0.052389738,0.109466705,0.166543673,0.213819545
Sample X4,0.083522629,0.189605074,0.311253964,0.397734217,0.498627847
Sample X5,0.082369559,0.200559239,0.328550014,0.434055924,0.526301528
Sample X6,0.099089075,0.198253099,0.314713174,0.425984434,0.529760738
Sample X7,0.147518017,0.307794754,0.481908331,0.634113577,0.789201499
Sample X8,0.182110118,0.315866244,0.462882675,0.622006342,0.782859614
Sample X9,0.177497838,0.322784664,0.47037763,0.577613145,0.735583742
Sample X10,0.277238397,0.495745172,0.737889882,0.917192274,1.091882387
Sample X11,0.262825022,0.46,0.70214471,0.879717498,1.011167483
Sample X12,0.279544537,0.557434419,0.709063131,0.873952148,1.111484578
"""

from io import StringIO
import pandas as pd
import numpy as np
from scipy import stats


s = StringIO(data)
example_df = pd.read_csv(s)
print(example_df)

          Time         0         5        10        15        20
0    Sample X1 -0.000075  0.071992  0.104278  0.173462  0.235728
1    Sample X2 -0.004687  0.065074  0.123304  0.174039  0.222468
2    Sample X3 -0.009300  0.052390  0.109467  0.166544  0.213820
3    Sample X4  0.083523  0.189605  0.311254  0.397734  0.498628
4    Sample X5  0.082370  0.200559  0.328550  0.434056  0.526302
5    Sample X6  0.099089  0.198253  0.314713  0.425984  0.529761
6    Sample X7  0.147518  0.307795  0.481908  0.634114  0.789201
7    Sample X8  0.182110  0.315866  0.462883  0.622006  0.782860
8    Sample X9  0.177498  0.322785  0.470378  0.577613  0.735584
9   Sample X10  0.277238  0.495745  0.737890  0.917192  1.091882
10  Sample X11  0.262825  0.460000  0.702145  0.879717  1.011167
11  Sample X12  0.279545  0.557434  0.709063  0.873952  1.111485


# **Description**

This function creates a linear fit of data starting with the complete set of data points (time, absorbance). If the initial linear fit is not of a high enough quality, the function iteratively reduces the maximum considered time value by 1 column and re-evaluates the quality of the fit. If no acceptable fit is found after iterative truncation of the domain, the function reduces the acceptable R^2 threshold by a set amount and tries again from the complete set of data points. The minimum number of data points allowed to be used for a regression is 5, and so the input file must contain 5 data points. If it does not, the function will not prompt you to save the output and will instead print an error message at the bottom of this page. The function iterates through the given number of rows in the y-value data. The function starts at a target R^2 of 0.99 by default.

# **Instructions**

1. Copy and paste raw data of interest into a new .xlsx (excel) file. This must first contain a column labeled 'Time' with a row for each sample name, followed by a column for each time point with a row for each sample's absorbance value at that time point. Be sure the input file is formatted the same way as the provided example, which will appear after you run this notebook (step 5).
2. Ensure you are connected to Colab runtime (click connect on top right of page)
3. Drag and drop your new .xlsx file into Colab's file manager (click folder on left side of screen, then drag and drop excel file).
4. Scroll down this page to the ‘User Inputs’ section to edit your run parameters. List the name of the Excel file you made, and provide the minimum acceptable R^2 and step size (respectively, these are 0.95 and 0.005 by default).
5. In the menu underneath the name of this notebook, click on Runtime, then Run all (if you do not see this menu, reveal it by clicking the arrow in the upper right hand corner of this window. When the run completes, Colab will prompt you to save the output excel file (Reminder: if you are not prompted to download the file, check for an error message at the bottom of this page. This indicates that the input data does not contain at least 5 points).
7. The output file contains 2 sheets: the first sheet describes the quality of the fits found for each sample, and the second sheet shows the truncated data that was used to calculate the rates.



In [ ]:
from io import StringIO
import pandas as pd
import numpy as np
from scipy import stats
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

#@title User inputs
filename = "go.xlsx" #@param {type:"string"}
R_min =  0.8#@param {type:"number"}
step = 0.005 #@param {type:"number"}

# R_min = 0.95
# step = 0.005

df = pd.read_excel(filename)
df_data = df.iloc[:,1:]
df_col_len = len(list(df_data.columns))
if df_col_len <= 4:
  print('error: check length of data')

df_t = df.T
df_t_new = df_t
df_t_new.columns = df_t_new.iloc[0]
df_t_new = df_t_new[1:].reset_index(drop=False)
df_t_new = df_t_new.rename(columns={'index':'Time'})
df_t_new.columns.name = None

time = df_t_new['Time']
startpoint = len(time)

R=0.99
R_list = [R]
R_new = R-step

while R_new >= R_min:
    R_list.append(R_new)
    R_new = R_new-step

outList_list = []

def trimmedRegression(R_list,time,abs,startpoint,sampleName):
    y = abs
    X = time
    y_len = len(y)
    X_len = len(X)
    if (y_len != X_len):
      print('error: check length of data')
    slope, intercept, r_value, p_value, std_err = stats.linregress(X,y)
    rsq_value = r_value**2
    numPts = len(y)
    outList = []
    outList = [sampleName,slope,rsq_value,numPts,R_list[0],list(X),list(y)]
    for R_thresh in R_list:
        j = startpoint-1
        while rsq_value < R_thresh and j>4:
            y_trunc = y[0:j]
            X_trunc = X[0:j]
            slope, intercept, r_value, p_value, std_err = stats.linregress(X_trunc,y_trunc)
            rsq_value = r_value**2
            numPts = len(y_trunc)
            outList = [sampleName,slope,rsq_value,numPts,R_thresh,list(X_trunc),list(y_trunc)]
            j = j-1
    if j == 4:
        outList = [sampleName,0,0,4,R_list[-1]]
    outList_list.append(outList)
    return(outList)

sampleDF = df_t_new.iloc[:,1:]
sampleColumnsList = list(sampleDF.columns)

for name in sampleColumnsList:
  abs = df_t_new[name].to_list()
  trimmedRegression(R_list,time,abs,startpoint,name)

outDF = pd.DataFrame(outList_list)
outDF = outDF.iloc[:,:5]
outDF.columns =['Sample', 'Slope', 'Calculated R^2', 'No. of data points', 'Target R^2']

df_toFill = df
df_toFill = df_toFill.set_index('Time')
df_toFill.iloc[:] = np.nan

for x in outList_list:
  sample_index = x[0]
  try:
    sample_time_list = x[5]
    sample_abs_list = x[6]
    for index in range(len(sample_time_list)):
      df_toFill.at[sample_index, sample_time_list[index]] = sample_abs_list[index]
  except:
    outstring = f"error: {sample_index} does not yield an acceptable model at R^2 threshold {R_min}"
    print(outstring)

outputStr = filename.replace('.xlsx','')+'_iterLinFit_output.xlsx'

with pd.ExcelWriter(outputStr) as writer:
    outDF.to_excel(writer, sheet_name='Fit_Statistics',index=False)
    df_toFill.to_excel(writer, sheet_name='Truncated_Data')

from google.colab import files
if df_col_len > 4:
  files.download(outputStr)

error: Bt2x does not yield an acceptable model at R^2 threshold 0.8
error: Bt2a does not yield an acceptable model at R^2 threshold 0.8
error: Pc1a does not yield an acceptable model at R^2 threshold 0.8
error: Eca does not yield an acceptable model at R^2 threshold 0.8
error: eta does not yield an acceptable model at R^2 threshold 0.8
error: Sample X28 does not yield an acceptable model at R^2 threshold 0.8
error: Sample X1 does not yield an acceptable model at R^2 threshold 0.8


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>